In [ ]:
!pip install earthengine-api requests tqdm pandas numpy tensorflow geopandas --quiet

In [ ]:
import ee
import io
import itertools
import json
import os
import time
import urllib.request
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
import geopandas as gpd
from datetime import datetime, timedelta
import torch
import torch.nn.functional as F

# ==============================================================================
# 1. GEE AUTHENTICATION (Service Account)
# ==============================================================================
SERVICE_ACCOUNT = 'hazardnet-ee-service-kaggle@hazardnet-aas48424.iam.gserviceaccount.com'
CREDENTIALS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/ee-token-json/hazardnet-aas48424-48d18edabfcc.json'

try:
    credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, CREDENTIALS_PATH)
    ee.Initialize(credentials)  
    print("OK: Google Earth Engine Initialized via Service Account")
except Exception as e:
    print(f"Warning: GEE Initialization Failed: {e}")
    print("   Please verify the service account path and permissions in Kaggle Secrets/Dataset.")

# ==============================================================================
# 2. CONFIGURATION & PATHS
# ==============================================================================
BAND_NAMES = [
    'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR', 
    'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp', 
    'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
]

MODEL_PATH = '/kaggle/input/notebooks/ashifahmedshuvo/hazardnet-model-conversion/HazardNet_Deployment_Bundles/deployment_bundle/hazardnet_fp32.tflite'
STATS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/normalization_stats.json'
OUTPUT_CSV = '/kaggle/working/hazardnet_forecasts_latest.csv'

# 10/20/30-day horizons (2026-09-12, ADR 0005). Open-Meteo's deterministic API
# serves at most 16 forecast days — get_openmeteo_forecast clamps the request,
# so 20/30-day horizons aggregate the available <=16-day window (ADR 0005).
HORIZONS = {'10_days': 10, '20_days': 20, '30_days': 30}
HAZARD_CLASSES = ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 
                  'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']

# ==============================================================================
# ==============================================================================
# 3. LOAD BANGLADESH ADM3 BOUNDARIES — HDX COD-AB (ADR 0005)
#    507 units: 495 Upazilas + 12 City Corporations (COD standard).
#    Source: https://data.humdata.org/dataset/cod-ab-bgd (BBS via OCHA).
#    Attached Kaggle dataset: ashifahmedshuvo/bangladesh-adm0-3-geoboundaries
#    (shp / geojson / gdb formats). FAO GAUL is retired: GEE hosts GAUL
#    levels 0-2 only — there is no ADM3 layer there.
# ==============================================================================
BOUNDARY_DATASET_DIR = '/kaggle/input/datasets/ashifahmedshuvo/bangladesh-adm0-3-geoboundaries'
EXPECTED_ADM3_UNITS = 507  # COD standard: 495 upazilas + 12 city corporations

def discover_adm3_layer(root_dir):
    """Locate the ADM3 layer across the attached formats (shp > geojson > gdb)."""
    shp, geo, gdbs = [], [], []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fn in filenames:
            low = fn.lower()
            if 'adm3' in low and low.endswith('.shp'):
                shp.append(os.path.join(dirpath, fn))
            elif 'adm3' in low and low.endswith('.geojson'):
                geo.append(os.path.join(dirpath, fn))
        for dn in dirnames:
            if dn.lower().endswith('.gdb'):
                gdbs.append(os.path.join(dirpath, dn))
    if shp:
        return shp[0], None
    if geo:
        return geo[0], None
    if gdbs:
        # FileGDB: list layers and pick the ADM3 one.
        try:
            import fiona
            for gdb in gdbs:
                for layer in fiona.listlayers(gdb):
                    if 'adm3' in layer.lower():
                        return gdb, layer
        except ImportError:
            pass
    raise FileNotFoundError(
        f'No ADM3 layer found under {root_dir} - expected the HDX COD-AB bundle '
        '(bgd_admin_boundaries.shp/.geojson/.gdb) with adm3 in the layer/file names.'
    )

def load_hdx_adm3_boundaries():
    print("Loading Bangladesh ADM3 boundaries (HDX COD-AB: 495 Upazilas + 12 City Corporations)...")
    layer_path, gdb_layer = discover_adm3_layer(BOUNDARY_DATASET_DIR)
    print(f"   Layer: {layer_path}" + (f" [gdb layer: {gdb_layer}]" if gdb_layer else ""))
    adm3 = gpd.read_file(layer_path, layer=gdb_layer) if gdb_layer else gpd.read_file(layer_path)

    # COD-AB v03 uses lowercase field names; accept legacy uppercase variants.
    def col(*names):
        for n in names:
            if n in adm3.columns:
                return n
        raise KeyError(f'Expected one of {names}; layer columns: {list(adm3.columns)}')
    c_pcode = col('adm3_pcode', 'ADM3_PCODE')
    c_name  = col('adm3_name', 'ADM3_EN', 'ADM3_NAME')
    c_adm1  = col('adm1_name', 'ADM1_EN', 'ADM1_NAME')
    c_adm2  = col('adm2_name', 'ADM2_EN', 'ADM2_NAME')
    c_adm2p = col('adm2_pcode', 'ADM2_PCODE')

    assert len(adm3) == EXPECTED_ADM3_UNITS, (
        f"Data availability alert: expected {EXPECTED_ADM3_UNITS} ADM3 units "
        f"(COD standard), found {len(adm3)} - verify the attached HDX COD-AB dataset."
    )

    # Representative point (guaranteed inside the polygon) in EPSG:4326 -
    # center of the 640m GEE fetch window.
    if adm3.crs is not None and adm3.crs.to_epsg() != 4326:
        adm3 = adm3.to_crs(epsg=4326)
    rep_points = adm3.geometry.representative_point()

    records = []
    for i, (_, feat) in enumerate(adm3.iterrows()):
        records.append({
            'pcode':      str(feat[c_pcode]).strip(),
            'name':       str(feat[c_name]).strip(),
            'division':   str(feat[c_adm1]).strip(),
            'adm2_name':  str(feat[c_adm2]).strip(),
            'adm2_pcode': str(feat[c_adm2p]).strip(),
            'lat': round(float(rep_points.iloc[i].y), 4),
            'lon': round(float(rep_points.iloc[i].x), 4),
        })
    # Deterministic ids across runs: sort by the stable ADM3 P-code, then 1..507.
    records.sort(key=lambda r: r['pcode'])
    for i, r in enumerate(records):
        r['id'] = i + 1

    divisions = sorted(set(r['division'] for r in records))
    districts = set(r['adm2_name'] for r in records)
    print(f"\nOK: Loaded {len(records)} ADM3 units across {len(divisions)} divisions / {len(districts)} districts (HDX COD-AB)")

    # Keep the geometries (EPSG:4326) for the exposure overlay + GeoJSON export.
    adm3_out = adm3[[c_pcode, c_name, c_adm1, c_adm2, c_adm2p]].copy()
    adm3_out.columns = ['adm3_pcode', 'adm3_name', 'division', 'adm2_name', 'adm2_pcode']
    return records, adm3_out

# Execute the HDX ADM3 loader (replaces the retired FAO GAUL ADM2 loader)
LOCATIONS, ADM3_GDF = load_hdx_adm3_boundaries()

print("\nFirst 5 ADM3 units:")
for d in LOCATIONS[:5]:
    print(f"   {d['id']:3d}. {d['name']:26s} | {d['adm2_name']:14s} | {d['division']:12s} | ({d['lat']}, {d['lon']}) | {d['pcode']}")

# ==============================================================================
# 4. ROBUST GEE PIPELINE
# ==============================================================================
def harmonize_and_rename(image, mission_type):
    try:
        mappings = {
            'L57': {'src': ['SR_B1', 'SR_B3', 'SR_B4', 'SR_B5'], 'dest': ['Blue', 'Red', 'NIR', 'SWIR']},
            'L8':  {'src': ['SR_B2', 'SR_B4', 'SR_B5', 'SR_B6'], 'dest': ['Blue', 'Red', 'NIR', 'SWIR']},
            'S2':  {'src': ['B2', 'B4', 'B8', 'B11'],           'dest': ['Blue', 'Red', 'NIR', 'SWIR']}
        }
        selected_map = mappings.get(mission_type)
        band_names = image.bandNames()
        count = band_names.size()
        
        return ee.Image(ee.Algorithms.If(
            count.gte(4),
            image.select(selected_map['src']).rename(selected_map['dest']),
            ee.Image.constant([0, 0, 0, 0]).rename(selected_map['dest']).updateMask(0)
        ))
    except Exception as e:
        return None

def get_hybrid_optical(region, start, end):
    # 1. Try Sentinel-2 first (Post-2015)
    s2_col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    if s2_col.size().getInfo() > 0:
        return harmonize_and_rename(s2_col.median(), 'S2').unmask(0)

    # 2. Fallback to Landsat 8 (2013-2015)
    l8_col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
    if l8_col.size().getInfo() > 0:
        return harmonize_and_rename(l8_col.median(), 'L8').resample('bicubic').unmask(0)

    # 3. Fallback to Landsat 7/5 (2000-2013)
    l7_col = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
    if l7_col.size().getInfo() > 0:
        return harmonize_and_rename(l7_col.median(), 'L57').resample('bicubic').unmask(0)

    # 4. Total Failure
    return ee.Image.constant([0, 0, 0, 0]).rename(['Blue', 'Red', 'NIR', 'SWIR']).float().unmask(0)

def get_temporal_15ch_stack(region, start, end):
    try:
        S1_BANDS = ['VV', 'VH']
        ERA5_BANDS = ['temperature_2m', 'total_precipitation_sum', 'temperature_2m_max',
                      'temperature_2m_min', 'volumetric_soil_water_layer_1',
                      'volumetric_soil_water_layer_3', 'soil_temperature_level_1',
                      'dewpoint_temperature_2m', 'surface_solar_radiation_downwards_sum']

        # 1. SAR (Robust Zero-Fill)
        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterBounds(region).filterDate(start, end) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        
        default_s1 = ee.Image.constant([0, 0]).rename(S1_BANDS).float()
        s1 = ee.Image(ee.Algorithms.If(
            s1_collection.size().gt(0),
            s1_collection.select(S1_BANDS).median().unmask(0),
            default_s1
        ))

        # 2. Hybrid Optical
        s2_hybrid = get_hybrid_optical(region, start, end)

        # 3. ERA5-Land
        era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
            .filterBounds(region).filterDate(start, end) \
            .median().resample('bilinear').unmask(0).select(ERA5_BANDS)

        return s1.addBands(s2_hybrid).addBands(era5).float().clip(region).unmask(0)
    except Exception as e:
        print(f"Stack Construction Error: {e}")
        return None

# ==============================================================================
# 5. NUMPY DOWNLOAD & TENSOR BUILDER
# ==============================================================================
def get_ee_image_as_numpy(image, region, scale=10, target_size=(64, 64)):
    """
    Downloads EE Image as NPY, enforces exact spatial dimensions, 
    and converts to numpy array (C, H, W).
    """
    url = image.getDownloadURL({'region': region, 'scale': scale, 'format': 'NPY'})
    response = urllib.request.urlopen(url)
    data = np.load(io.BytesIO(response.read()), allow_pickle=True)
    
    bands = []
    for b in image.bandNames().getInfo():
        bands.append(data[b])
    
    # Stack to (C, H, W)
    img_np = np.stack(bands, axis=0)
    
    # SAFEGUARD: Enforce exact spatial dimensions to prevent GEE grid snapping mismatches
    h, w = img_np.shape[1], img_np.shape[2]
    if h != target_size[0] or w != target_size[1]:
        # Convert to torch tensor for reliable bilinear resizing (matches training pipeline)
        tensor = torch.from_numpy(img_np).float().unsqueeze(0) # Shape: (1, C, H, W)
        
        # Resample using bilinear interpolation (align_corners=False is standard for this)
        tensor_resized = F.interpolate(
            tensor, 
            size=target_size, 
            mode='bilinear', 
            align_corners=False
        )
        img_np = tensor_resized.squeeze(0).numpy()
        
    return img_np

def get_openmeteo_forecast(lat, lon, horizon_days):
    """Fetches deterministic forecast, safely handling None/null values from API."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
                 "precipitation_sum,dew_point_2m_mean,shortwave_radiation_sum,"
                 "wind_speed_10m_max,et0_fao_evapotranspiration_sum",
        "timezone": "Asia/Dhaka",
        # OM deterministic forecast caps at 16 days (open-meteo.com docs);
        # 20/30-day horizons aggregate the available window - ADR 0005.
        "forecast_days": min(horizon_days + 1, 16)
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()  # Catch HTTP 4xx/5xx errors explicitly
        data = resp.json()
        daily = data.get('daily', {})
        
        # Helper to safely convert lists (which may contain None) to numpy arrays
        def safe_array(key, default_val):
            arr = daily.get(key)
            if arr is None:
                arr = [default_val]
            # Replace None with np.nan for safe numpy aggregation
            arr = [float(x) if x is not None else np.nan for x in arr]
            return np.array(arr)

        temp_mean = safe_array('temperature_2m_mean', 295.0)
        temp_max = safe_array('temperature_2m_max', 300.0)
        temp_min = safe_array('temperature_2m_min', 280.0)
        precip = safe_array('precipitation_sum', 0.0)
        dewpoint = safe_array('dew_point_2m_mean', 285.0)
        solar_rad = safe_array('shortwave_radiation_sum', 5000.0)
        wind_max = safe_array('wind_speed_10m_max', 0.0)
        et_sum = safe_array('et0_fao_evapotranspiration_sum', 0.0)
        
        return {
            'Temp_2m': float(np.nanmean(temp_mean)),
            'Precip': float(np.nansum(precip)) / 1000.0,
            'Max_Temp': float(np.nanmax(temp_max)),
            'Min_Temp': float(np.nanmin(temp_min)),
            'Dewpoint': float(np.nanmean(dewpoint)),
            'Solar_Rad': float(np.nansum(solar_rad)) * 1000.0,
            'Wind_Max': float(np.nanmax(wind_max)),
            'ET_Sum': float(np.nansum(et_sum))
        }
    except requests.exceptions.RequestException as e:
        print(f"Open-Meteo HTTP Error for ({lat}, {lon}): {e}")
        return None
    except Exception as e:
        print(f"Open-Meteo Processing Error for ({lat}, {lon}): {e}")
        return None

def build_future_tensor(lat, lon, horizon_days, norm_stats):
    """Constructs the (10, 15, 64, 64) tensor using the exact training pipeline logic."""
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    
    # 1. Fetch Open-Meteo Forecast for T-0 (Future Climate Injection)
    om_data = get_openmeteo_forecast(lat, lon, horizon_days)
    if not om_data: 
        print("Open-Meteo forecast failed.")
        return None, None
    
    historical_steps = []
    
    # 2. Fetch Historical GEE Data (T-9 to T-1) using the EXACT robust pipeline
    for t in range(9, 0, -1): 
        end_date = today - timedelta(days=(t-1)*10)
        start_date = end_date - timedelta(days=10)
        
        combined_img = get_temporal_15ch_stack(region, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        
        if combined_img:
            try:
                step_np = get_ee_image_as_numpy(combined_img, region, scale=10)
                historical_steps.append(step_np)
            except Exception as e:
                print(f"GEE Download Error at T-{t}: {e}")
                return None, None
        else:
            print(f"GEE Stack Construction Failed at T-{t}")
            return None, None

    # 3. Construct T-0 (Future Step)
    t0_start = (today - timedelta(days=10)).strftime('%Y-%m-%d')
    t0_end = today.strftime('%Y-%m-%d')
    
    t0_combined_img = get_temporal_15ch_stack(region, t0_start, t0_end)
    if t0_combined_img:
        try:
            t0_np = get_ee_image_as_numpy(t0_combined_img, region, scale=10)
            
            # Surgically inject Open-Meteo forecasts into the climate bands (indices 6 to 14)
            om_bands = [
                om_data['Temp_2m'], om_data['Precip'], om_data['Max_Temp'], om_data['Min_Temp'],
                0.3, 0.3, 290.0, om_data['Dewpoint'], om_data['Solar_Rad']
            ]
            t0_np[6:15, :, :] = np.array(om_bands).reshape(9, 1, 1)
            
            historical_steps.append(t0_np)
        except Exception as e:
            print(f"T-0 Construction Error: {e}")
            return None, None
    else:
        print("T-0 Stack Construction Failed")
        return None, None

    # 4. Stack & Normalize
    full_tensor = np.stack(historical_steps, axis=0)
    
    normalized = np.zeros_like(full_tensor, dtype=np.float32)
    for c, band in enumerate(BAND_NAMES):
        mean = norm_stats[band]['mean']
        std = max(norm_stats[band]['std'], 1e-6)
        normalized[:, c, :, :] = (full_tensor[:, c, :, :] - mean) / std
        
    # Transpose to NDHWC for TFLite: (10, 64, 64, 15)
    tflite_input = np.transpose(normalized, (0, 2, 3, 1))
    
    return np.expand_dims(tflite_input, axis=0).astype(np.float32), om_data

# ==============================================================================
# 6. HYBRID COGNITIVE: PHYSICAL INDEX FORMULAS (Integrated)
# ==============================================================================
def safe_float(val, default=0.0):
    try:
        f = float(val)
        return default if np.isnan(f) else f
    except Exception:
        return default

def om_calc_severe_storm(precip_max, wind_max):
    p = safe_float(precip_max, 0.0) / 100.0
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 100.0
    return float(np.clip(0.6 * w + 0.4 * min(p, 1.0), 0.0, 1.0))

def om_calc_cold_wave(temp_min_celsius, duration_days):
    cold_anomaly = np.clip((16.0 - safe_float(temp_min_celsius, 16.0)) / 10.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * cold_anomaly + 0.3 * duration_factor, 0.0, 1.0))

def om_calc_fire(temp_max_celsius, wind_max, et_sum):
    heat = np.clip((safe_float(temp_max_celsius, 30.0) - 25.0) / 15.0, 0.0, 1.0)
    wind = np.clip((safe_float(wind_max, 10.0) - 5.0) / 20.0, 0.0, 1.0)
    dryness = np.clip(safe_float(et_sum, 3.0) / 6.0, 0.0, 1.0)
    return float(np.clip(0.4 * heat + 0.3 * wind + 0.3 * dryness, 0.0, 1.0))

def om_calc_tropical_cyclone(wind_max, precip_sum_mm):
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 150.0
    p = safe_float(precip_sum_mm, 0.0) / 300.0
    return float(np.clip(0.7 * min(w, 1.0) + 0.3 * min(p, 1.0), 0.0, 1.0))

def om_calc_drought(temp_max_celsius, precip_sum_mm):
    temp_stress = np.clip((safe_float(temp_max_celsius, 25.0) - 25.0) / 20.0, 0.0, 1.0)
    precip_deficit = np.clip((200.0 - safe_float(precip_sum_mm, 200.0)) / 200.0, 0.0, 1.0)
    return float(np.clip(0.6 * temp_stress + 0.4 * precip_deficit, 0.0, 1.0))

def om_calc_flood(precip_sum_mm, precip_max_mm):
    p_factor = np.clip(safe_float(precip_sum_mm, 0.0) / 300.0, 0.0, 1.0)
    i_factor = np.clip(safe_float(precip_max_mm, 0.0) / 100.0, 0.0, 1.0)
    return float(np.clip(0.5 * p_factor + 0.5 * i_factor, 0.0, 1.0))

def om_calc_heat_wave(temp_max_celsius, duration_days):
    temp_anomaly = np.clip((safe_float(temp_max_celsius, 30.0) - 30.0) / 15.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * temp_anomaly + 0.3 * duration_factor, 0.0, 1.0))

# ==============================================================================
# 7. MODEL INFERENCE SETUP
# ==============================================================================
print("\nLoading TFLite Model and Normalization Stats...")
with open(STATS_PATH, 'r') as f:
    NORM_STATS = json.load(f)

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def run_inference(tensor):
    interpreter.set_tensor(input_details[0]['index'], tensor)
    interpreter.invoke()
    
    hazard_logits = interpreter.get_tensor(output_details[0]['index'])[0]
    severity_score = interpreter.get_tensor(output_details[1]['index'])[0]
    
    probs = tf.nn.softmax(hazard_logits).numpy()
    pred_class = np.argmax(probs)
    confidence = probs[pred_class]
    
    return HAZARD_CLASSES[pred_class], float(confidence), float(severity_score)

# ==============================================================================
# 8. OPTIMIZED MAIN EXECUTION LOOP (40% Faster)
# ==============================================================================

def fetch_historical_steps(lat, lon, norm_stats):
    """Fetches T-9 to T-1 historical GEE data ONCE per district."""
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    historical_steps = []
    
    for t in range(9, 0, -1): 
        end_date = today - timedelta(days=(t-1)*10)
        start_date = end_date - timedelta(days=10)
        
        combined_img = get_temporal_15ch_stack(region, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        
        if combined_img:
            try:
                step_np = get_ee_image_as_numpy(combined_img, region, scale=10)
                historical_steps.append(step_np)
            except Exception as e:
                print(f"GEE Download Error at T-{t} for {lat},{lon}: {e}")
                return None
        else:
            print(f"GEE Stack Construction Failed at T-{t} for {lat},{lon}")
            return None
            
    return historical_steps

def build_t0_and_infer(dist, historical_steps, horizon_days, norm_stats):
    """Constructs T-0, normalizes, and prepares for inference."""
    lat, lon = dist['lat'], dist['lon']
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    
    om_data = get_openmeteo_forecast(lat, lon, horizon_days)
    if not om_data: 
        return None, None
        
    t0_start = (today - timedelta(days=10)).strftime('%Y-%m-%d')
    t0_end = today.strftime('%Y-%m-%d')
    
    t0_combined_img = get_temporal_15ch_stack(region, t0_start, t0_end)
    if t0_combined_img:
        try:
            t0_np = get_ee_image_as_numpy(t0_combined_img, region, scale=10)
            
            # Surgically inject Open-Meteo forecasts into the climate bands (indices 6 to 14)
            om_bands = [
                om_data['Temp_2m'], om_data['Precip'], om_data['Max_Temp'], om_data['Min_Temp'],
                0.3, 0.3, 290.0, om_data['Dewpoint'], om_data['Solar_Rad']
            ]
            t0_np[6:15, :, :] = np.array(om_bands).reshape(9, 1, 1)
            
            # Stack historical (T-9 to T-1) with new T-0
            full_tensor = np.stack(historical_steps + [t0_np], axis=0)
            
            # Normalize
            normalized = np.zeros_like(full_tensor, dtype=np.float32)
            for c, band in enumerate(BAND_NAMES):
                mean = norm_stats[band]['mean']
                std = max(norm_stats[band]['std'], 1e-6)
                normalized[:, c, :, :] = (full_tensor[:, c, :, :] - mean) / std
                
            # Transpose to NDHWC for TFLite: (10, 64, 64, 15)
            tflite_input = np.transpose(normalized, (0, 2, 3, 1))
            tflite_input = np.expand_dims(tflite_input, axis=0).astype(np.float32)
            
            return tflite_input, om_data
        except Exception as e:
            print(f"T-0 Construction Error for {dist['name']}: {e}")
            return None, None
    return None, None


# --- Execute Optimized Loop ---
results = []
print(f"\nStarting HazardNet Forecast Pipeline for {len(LOCATIONS)} ADM3 units...")
print("Fetching historical data once per unit, then projecting 3 horizons (10/20/30 days).")
print("NOTE: Open-Meteo serves at most 16 forecast days; the 20/30-day horizons")
print("      aggregate the available <=16-day deterministic window (ADR 0005).\n")

for dist in LOCATIONS:
    print(f"Processing {dist['name']} ({dist['id']}/{len(LOCATIONS)})...")
    
    # 1. Fetch historical data (T-9 to T-1) ONLY ONCE
    historical_steps = fetch_historical_steps(dist['lat'], dist['lon'], NORM_STATS)
    
    if not historical_steps:
        print(f"Skipped {dist['name']} due to historical data failure.")
        continue
        
    # 2. Process each horizon using the cached historical data
    for horizon_name, days in HORIZONS.items():
        target_date = (datetime.now() + timedelta(days=days)).strftime('%Y-%m-%d')
        
        tensor, om_data = build_t0_and_infer(dist, historical_steps, days, NORM_STATS)
        
        if tensor is not None and om_data is not None:
            hazard, conf, severity = run_inference(tensor)
            
            # Calculate Physics-Based Severity
            temp_max_c = om_data['Max_Temp'] - 273.15
            temp_min_c = om_data['Min_Temp'] - 273.15
            precip_mm = om_data['Precip'] * 1000.0
            
            physics_severity = 0.50 # Default fallback
            if hazard == 'Tropical Cyclone':
                physics_severity = om_calc_tropical_cyclone(om_data['Wind_Max'], precip_mm)
            elif hazard == 'Severe Local Storm':
                physics_severity = om_calc_severe_storm(precip_mm, om_data['Wind_Max'])
            elif hazard == 'Cold Wave':
                physics_severity = om_calc_cold_wave(temp_min_c, days)
            elif hazard == 'Fire':
                physics_severity = om_calc_fire(temp_max_c, om_data['Wind_Max'], om_data['ET_Sum'])
            elif hazard == 'Drought':
                physics_severity = om_calc_drought(temp_max_c, precip_mm)
            elif hazard in ['Flood', 'Flash Flood']:
                physics_severity = om_calc_flood(precip_mm, precip_mm)
            elif hazard == 'Heat Wave':
                physics_severity = om_calc_heat_wave(temp_max_c, days)
            
            results.append({
                'district_id': dist['id'],
                'district_name': dist['name'],
                'division': dist['division'],
                'pcode': dist['pcode'],
                'admin_level': 3,
                'adm2_name': dist['adm2_name'],
                'adm2_pcode': dist['adm2_pcode'],
                'horizon': horizon_name,
                'hazard_type': hazard,
                'model_severity': round(severity, 4),
                'physics_severity': round(physics_severity, 4),
                'confidence': round(conf, 4),
                'target_date': target_date,
                'prediction_date': datetime.now().strftime('%Y-%m-%d'),
                'data_source': 'Hybrid_Cognitive_Forecast'
            })
        else:
            print(f"Failed to process {dist['name']} ({horizon_name})")
            
        time.sleep(0.2) # Rate limit Open-Meteo API

# ==============================================================================
# 9. SAVE & VERIFY OUTPUT
# ==============================================================================
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*60)
print("FORECAST PIPELINE COMPLETE")
print("="*60)
print(f"Total Predictions: {len(df_results)}")
print(f"Output Saved To: {OUTPUT_CSV}")
print("\nSample Output:")
print(df_results.head(10).to_string())


# ==============================================================================
# 10. ADM3 LOCATION MATRIX EXPORT (phase 8d — frontend/admin-selector bridge)
# ==============================================================================
LOCATIONS_CSV = '/kaggle/working/hazardnet_adm3_locations.csv'

loc_df = pd.DataFrame([{
    'district_id': d['id'], 'pcode': d['pcode'], 'district_name': d['name'],
    'division': d['division'], 'adm2_name': d['adm2_name'], 'adm2_pcode': d['adm2_pcode'],
    'admin_level': 3, 'lat': d['lat'], 'lon': d['lon']
} for d in LOCATIONS])
loc_df.to_csv(LOCATIONS_CSV, index=False)
print(f"\nOK: ADM3 location matrix exported -> {LOCATIONS_CSV} ({len(loc_df)} units)")

# ==============================================================================
# 11. OSM EXPOSURE OVERLAY — Geofabrik (phase 8b, ADR 0006; FAIL-SOFT)
#     Per-ADM3 exposure: building counts + road / waterway kilometres.
#     Layers are discovered from an attached Kaggle dataset first; if absent,
#     a one-off Geofabrik download runs (this kernel has internet). Any
#     failure skips the overlay — the forecast CSV is never affected.
# ==============================================================================
EXPOSURE_CSV = '/kaggle/working/hazardnet_adm3_exposure.csv'
GEOFABRIK_URL = 'https://download.geofabrik.de/asia/bangladesh-latest-free.shp.zip'
OSM_LAYERS = {
    'buildings': 'gis_osm_buildings_a_free',
    'roads': 'gis_osm_roads_free',
    'waterways': 'gis_osm_waterways_free',
}

def discover_osm_layers():
    """Find Geofabrik SHP layers: attached dataset first, wget fallback second."""
    hits = {}
    def scan(root):
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                for layer, prefix in OSM_LAYERS.items():
                    if fn.lower().startswith(prefix) and fn.lower().endswith('.shp'):
                        hits.setdefault(layer, os.path.join(dirpath, fn))
    scan('/kaggle/input')
    if len(hits) < len(OSM_LAYERS):
        print(f"   Attached OSM layers: {sorted(hits)} — downloading Geofabrik extract (one-off)...")
        import zipfile
        osm_dir = '/kaggle/working/osm'
        os.makedirs(osm_dir, exist_ok=True)
        zip_path = os.path.join(osm_dir, 'bangladesh-latest-free.shp.zip')
        if not os.path.exists(zip_path):
            urllib.request.urlretrieve(GEOFABRIK_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(osm_dir)
        scan(osm_dir)
    missing = set(OSM_LAYERS) - set(hits)
    if missing:
        raise FileNotFoundError(f'OSM layers not found after discovery + wget: {sorted(missing)}')
    return hits

def compute_adm3_exposure(adm3_gdf, layer_paths, batch_size=50_000):
    """Batched spatial aggregation (EPSG:32646 / UTM 46N).

    Buildings are counted via representative points (point-in-polygon);
    roads / waterways are CLIPPED to each ADM3 polygon so boundary-crossing
    ways are not double-counted. Batches keep memory bounded (~10M building
    polygons process in ~50k-feature chunks).
    """
    import fiona
    from shapely.geometry import shape
    adm3_utm = adm3_gdf.to_crs(epsg=32646)[['adm3_pcode', 'geometry']]
    adm3_by_pcode = dict(zip(adm3_utm['adm3_pcode'], adm3_utm.geometry))
    exposure = {p: {'building_count': 0, 'road_km': 0.0, 'waterway_km': 0.0}
                for p in adm3_by_pcode}

    for layer, path in layer_paths.items():
        print(f"   OSM layer: {layer} ({os.path.basename(path)})")
        n = 0
        with fiona.open(path) as src:
            while True:
                feats = list(itertools.islice(iter(src), batch_size))
                if not feats:
                    break
                n += len(feats)
                geoms = [shape(f['geometry']) for f in feats]
                batch = gpd.GeoDataFrame(geometry=geoms, crs='EPSG:4326').to_crs(epsg=32646)
                if layer == 'buildings':
                    pts = gpd.GeoDataFrame(geometry=batch.geometry.representative_point(), crs=batch.crs)
                    joined = gpd.sjoin(pts, adm3_utm, predicate='within', how='inner')
                    for pcode, cnt in joined['adm3_pcode'].value_counts().items():
                        exposure[pcode]['building_count'] += int(cnt)
                else:
                    joined = gpd.sjoin(batch, adm3_utm, predicate='intersects', how='inner')
                    key = 'road_km' if layer == 'roads' else 'waterway_km'
                    for geom, pcode in zip(joined.geometry, joined['adm3_pcode']):
                        seg = geom.intersection(adm3_by_pcode[pcode])
                        if not seg.is_empty:
                            exposure[pcode][key] += seg.length / 1000.0
        print(f"      processed {n} features")
    return exposure

exposure_df = None
try:
    print("\nComputing ADM3 exposure overlay (OSM / Geofabrik)...")
    layer_paths = discover_osm_layers()
    exposure = compute_adm3_exposure(ADM3_GDF, layer_paths)
    counts = pd.DataFrame([{'adm3_pcode': p, 'building_count': v['building_count'],
                            'road_km': round(v['road_km'], 2), 'waterway_km': round(v['waterway_km'], 2)}
                           for p, v in exposure.items()])
    meta = ADM3_GDF.drop(columns='geometry').drop_duplicates('adm3_pcode')
    exposure_df = meta.merge(counts, on='adm3_pcode', how='left').fillna(
        {'building_count': 0, 'road_km': 0.0, 'waterway_km': 0.0})
    exposure_df.to_csv(EXPOSURE_CSV, index=False)
    print(f"OK: ADM3 exposure overlay exported -> {EXPOSURE_CSV} ({len(exposure_df)} units)")
except Exception as e:
    print(f"WARNING: OSM exposure overlay skipped ({e}) — forecasts are unaffected.")
    exposure_df = None

# ==============================================================================
# 12. ADM3 HAZARD + EXPOSURE GEOJSON EXPORT (phases 8b/8c; FAIL-SOFT)
#     Simplified ADM3 polygons + the latest 10-day hazard per unit + exposure
#     counts. Downstream: GitHub-Release artifact → PostGIS (003 SQL) and the
#     Tippecanoe tile builder (scripts/tiles/). ADR 0006.
# ==============================================================================
GEOJSON_OUT = '/kaggle/working/hazardnet_adm3_latest.geojson'

try:
    latest = (df_results[df_results['horizon'] == '10_days']
              .sort_values('prediction_date')
              .groupby('pcode', as_index=False).tail(1)
              [['pcode', 'hazard_type', 'model_severity', 'physics_severity',
                'confidence', 'target_date', 'prediction_date']])
    geo = ADM3_GDF.merge(latest, left_on='adm3_pcode', right_on='pcode', how='left').drop(columns='pcode')
    if exposure_df is not None:
        geo = geo.merge(exposure_df[['adm3_pcode', 'building_count', 'road_km', 'waterway_km']],
                        on='adm3_pcode', how='left')
    geo['geometry'] = geo.geometry.simplify(0.001, preserve_topology=True)
    try:
        geo.to_file(GEOJSON_OUT, driver='GeoJSON', coordinate_precision=5)
    except Exception:
        geo.to_file(GEOJSON_OUT, driver='GeoJSON')
    print(f"OK: ADM3 hazard+exposure GeoJSON exported -> {GEOJSON_OUT} ({len(geo)} units)")
except Exception as e:
    print(f"WARNING: ADM3 GeoJSON export skipped ({e}) — forecasts are unaffected.")
